# JWHT — Evaluation Notebook

**Joint** Walsh-Hadamard Transform model with learnable decimation.  
Loads checkpoint **`JWHT_best_model.pth`**.  
Computes all paper metrics (MACE, CVar, CSD, NAACC, RMSEP25S) and generates figures.  
Includes fault-order estimation (Table 4, ε = 0.0645 %).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FuncFormatter
from torch.utils.data import DataLoader, TensorDataset
from scipy.interpolate import interp1d
from scipy.fft import rfft, rfftfreq
from scipy.stats import circmean, circvar, circstd
import pandas as pd
import seaborn as sns
import random, os

from models import JWHT

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

def set_seed(seed=42):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed);    random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load data and checkpoint

In [ ]:
DATA_DIR = '.'
CKPT_DIR = 'checkpoints'
SR = 50000        # Safran dataset: 50 kHz  (change to 1500 for synthetic)

x_test      = np.load(os.path.join(DATA_DIR, 'x_test.npy'))   # (N, 1024)
y_test      = np.load(os.path.join(DATA_DIR, 'y_test.npy'))   # (N, 32)
y_test_real = y_test[:, :16]
y_test_imag = y_test[:, 16:]

x_t    = torch.tensor(x_test, dtype=torch.float32)
loader = DataLoader(TensorDataset(x_t), batch_size=64, shuffle=False)

# ── JWHT: single joint model ──────────────────────────────────────────────────
model = JWHT().to(device)
model.load_state_dict(torch.load(
    os.path.join(CKPT_DIR, 'JWHT_best_model.pth'), map_location=device, weights_only=False))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'JWHT loaded  |  parameters: {n_params:,}')
print('filter1 weights:', model.downsample1.filter1.weight.data)
print('filter2 weights:', model.downsample1.filter2.weight.data)

## 2. Run inference

In [ ]:
all_preds = []
with torch.no_grad():
    for (x,) in loader:
        all_preds.append(model(x.to(device)).cpu().numpy())

preds      = np.concatenate(all_preds)        # (N, 32)
real_pred  = preds[:, :16]
imag_pred  = preds[:, 16:]
phase_pred = np.arctan2(imag_pred,  real_pred)
phase_true = np.arctan2(y_test_imag, y_test_real)
print('phase_pred shape:', phase_pred.shape)

## 3. Evaluation metrics

In [ ]:
def mace(true, pred):
    return np.degrees(np.mean(np.abs(np.angle(np.exp(1j*(pred-true))))))

def cvar(true, pred):
    d = np.angle(np.exp(1j*(pred-true)))
    return 1 - np.abs(np.mean(np.exp(1j*d)))

def csd(true, pred):
    d = np.angle(np.exp(1j*(pred-true)))
    return np.degrees(np.sqrt(-2*np.log(np.abs(np.mean(np.exp(1j*d))))))

def naacc(true, pred):
    d = np.degrees(np.abs(np.angle(np.exp(1j*(pred-true)))))
    return 100*(1 - np.mean(d)/180)

def rmsep25s(true, pred, sr):
    d = ((np.degrees(pred)-np.degrees(true))+180)%360-180
    d = d.reshape(-1)
    win = int(sr*0.25)
    return np.mean([np.sqrt(np.mean(d[i*win:(i+1)*win]**2)) for i in range(len(d)//win)])

print(f'MACE     : {mace(phase_true, phase_pred):.4f} °')
print(f'CVar     : {cvar(phase_true, phase_pred):.2e}')
print(f'CSD      : {csd(phase_true, phase_pred):.4f} °')
print(f'NAACC    : {naacc(phase_true, phase_pred):.4f} %')
print(f'RMSEP25S : {rmsep25s(phase_true, phase_pred, SR):.4f} °')

## 4. Phase comparison plot

In [ ]:
def pi_fmt(v, _):
    m = {0:'0', np.pi:r'$\pi$', -np.pi:r'$-\pi$',
         np.pi/2:r'$\pi/2$', -np.pi/2:r'$-\pi/2$'}
    return m.get(round(v,6), f'{v:.2f}')

p_true  = phase_true.reshape(-1)
p_pred  = phase_pred.reshape(-1)
t       = np.arange(len(p_true)) / SR
seg     = slice(0, min(int(0.01*SR), len(p_true)))   # short window for Safran

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[seg], p_true[seg], color='#E26472', label='True Phase', lw=2, alpha=0.8)
ax.plot(t[seg], p_pred[seg], color='#2B9CD7', label='Predicted Phase JWHT',
        lw=2, ls='--', alpha=0.8)
ax.set_xlabel('Time (seconds)', fontsize=20, fontfamily='serif')
ax.set_ylabel('Phase (radians)', fontsize=20, fontfamily='serif')
ax.yaxis.set_major_locator(MultipleLocator(np.pi/2))
ax.yaxis.set_major_formatter(FuncFormatter(pi_fmt))
ax.legend(fontsize=16, framealpha=0.8); ax.grid(False)
plt.tight_layout()
plt.savefig('JWHT_phase_comparison.svg', format='svg', bbox_inches='tight')
plt.show()

## 5. Rose diagrams (0°, 90°, 180°, 270°)

In [ ]:
def rose_diagram(true_phases, pred_phases, target_deg=None, num_bins=180, title=''):
    p_t    = true_phases % (2*np.pi)
    errors = np.angle(np.exp(1j*(pred_phases - true_phases)))
    if target_deg is not None:
        tr   = np.radians(target_deg)
        mask = np.abs(np.angle(np.exp(1j*(p_t-tr)))) < np.radians(5)
        p_t, errors = p_t[mask], errors[mask]
    bins   = np.linspace(0, 2*np.pi, num_bins+1)
    ctrs   = (bins[:-1]+bins[1:])/2
    w      = 2*np.pi/num_bins
    idx    = np.digitize(p_t, bins)-1
    binn   = [np.mean(np.abs(errors[idx==i])) if np.any(idx==i) else 0 for i in range(num_bins)]
    me     = np.degrees(np.mean(np.abs(errors)))
    se     = np.degrees(np.std(np.abs(errors)))
    fig,ax = plt.subplots(subplot_kw={'projection':'polar'}, figsize=(5,5))
    ax.bar(ctrs, binn, width=w, color='none', edgecolor='#9d84bf', linewidth=1.5, alpha=0.8)
    ax.set_title(f'{title}\nMean Error: {me:.2f}°, Std: {se:.2f}°', va='bottom', fontsize=11)
    plt.tight_layout(); plt.show()

pf_true = phase_true.reshape(-1)
pf_pred = phase_pred.reshape(-1)
for deg in [0, 90, 180, 270]:
    rose_diagram(pf_true, pf_pred, target_deg=deg, title=f'JWHT — Phase: {deg}°')

## 6. Fault-order estimation (Table 4)

Angular resampling → order spectrum → locate spectral peak near theoretical order k_th = 23.277.  
Expected result: k_est = 23.29200, ε = 0.0645 % (Table 4 in paper).

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
K_TH          = 7.759 * 3    # theoretical fault order = 23.277
DELTA         = 0.5          # search half-width around k_th
POINTS_PER_REV = 360         # angular resolution

# ── helpers ───────────────────────────────────────────────────────────────────
def equal_angle_resample(signal, pred_phase, points_per_rev=360):
    """Resample signal onto equal-angle grid using predicted phase."""
    phase_deg = np.rad2deg(np.unwrap(pred_phase))
    total_deg = phase_deg[-1] - phase_deg[0]
    n_equal   = int(total_deg / 360 * points_per_rev)
    phase_eq  = np.linspace(phase_deg[0], phase_deg[-1], n_equal)
    sig_eq    = interp1d(
        phase_deg, signal, kind='linear',
        bounds_error=False, fill_value='extrapolate'
    )(phase_eq)
    return sig_eq

def order_error(angle_signal, k_theory, points_per_rev=360, search_band=0.5):
    """Estimate dominant order near k_theory and return relative error ε (%)."""
    spectrum = np.abs(rfft(angle_signal))
    orders   = rfftfreq(len(angle_signal), d=1/points_per_rev)
    band     = (orders >= k_theory-search_band) & (orders <= k_theory+search_band)
    k_est    = orders[band][np.argmax(spectrum[band])]
    eps      = abs(k_est - k_theory) / k_theory * 100
    return k_est, eps

# ── run ───────────────────────────────────────────────────────────────────────
# raw_signal: use real_pred (the reconstructed real part) as proxy vibration signal
raw_signal = real_pred.reshape(-1)
pred_phase_flat = phase_pred.reshape(-1)

angle_signal = equal_angle_resample(raw_signal, pred_phase_flat, POINTS_PER_REV)
k_est, eps   = order_error(angle_signal, K_TH, POINTS_PER_REV, DELTA)

print(f'Theoretical order  k_th  = {K_TH:.3f}')
print(f'Estimated order    k_est = {k_est:.5f}')
print(f'Relative error     ε     = {eps:.4f} %')

## 7. Circular statistics summary

In [ ]:
p_true_flat = phase_true.reshape(-1)
p_pred_flat = phase_pred.reshape(-1)

# Analytic signal phase errors
analytic_true = y_test_real.reshape(-1) + 1j*y_test_imag.reshape(-1)
analytic_pred = real_pred.reshape(-1)   + 1j*imag_pred.reshape(-1)
phase_diff    = np.abs(np.angle(analytic_true) - np.angle(analytic_pred))

c_mean = circmean(np.rad2deg(phase_diff), high=180, low=-180)
c_var  = circvar(np.rad2deg(phase_diff),  high=180, low=-180)
c_std  = circstd(np.rad2deg(phase_diff),  high=180, low=-180)
acc    = 1 - (c_mean / 180)

print(f'Circular Mean  : {c_mean:.4f} °')
print(f'Circular Var   : {c_var:.6f}')
print(f'Circular Std   : {c_std:.4f} °')
print(f'NAACC (ACC)    : {acc:.6f}')